In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Import dataset for initial inspection

In [2]:
covid_data = pd.read_csv("../data/raw/WHO-COVID-19-global-data.csv")

## Initial inspection 
- First/last rows
- Shape of dataset  
- Column names and dtypes
- Describe 

In [3]:
covid_data.head()

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
0,2020-01-05,AI,Anguilla,AMRO,NaN,0,NaN,0
1,2020-01-12,AI,Anguilla,AMRO,NaN,0,NaN,0
2,2020-01-19,AI,Anguilla,AMRO,NaN,0,NaN,0
3,2020-01-26,AI,Anguilla,AMRO,NaN,0,NaN,0
4,2020-02-02,AI,Anguilla,AMRO,NaN,0,NaN,0


In [4]:
covid_data.tail()

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
77515,2026-02-08,XXL,International commercial vessel,OTHER,NaN,29,NaN,0
77516,2026-02-15,XXL,International commercial vessel,OTHER,NaN,29,NaN,0
77517,2026-02-22,XXL,International commercial vessel,OTHER,NaN,29,NaN,0
77518,2026-03-01,XXL,International commercial vessel,OTHER,NaN,29,NaN,0
77519,2026-03-08,XXL,International commercial vessel,OTHER,NaN,29,NaN,0


In [5]:
covid_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77520 entries, 0 to 77519
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date_reported      77520 non-null  object 
 1   Country_code       77197 non-null  object 
 2   Country            77520 non-null  object 
 3   WHO_region         71706 non-null  object 
 4   New_cases          51020 non-null  float64
 5   Cumulative_cases   77520 non-null  int64  
 6   New_deaths         36608 non-null  float64
 7   Cumulative_deaths  77520 non-null  int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 4.7+ MB


In [6]:
covid_data.describe().T

,count,mean,std,min,25%,50%,75%,max
New_cases,51020.0,1.527194e+04,2.369791e+05,-65079.0,8.0,150.0,2017.0,40475477.0
Cumulative_cases,77520.0,2.160477e+06,8.834618e+06,0.0,7612.5,63993.0,696614.0,103436829.0
New_deaths,36608.0,1.942998e+02,1.012255e+03,-3432.0,0.0,5.0,50.0,47687.0
Cumulative_deaths,77520.0,2.243399e+04,8.905065e+04,0.0,46.0,743.0,8530.0,1236353.0


In [7]:
covid_data[covid_data["New_cases"]==covid_data["New_cases"].max()]

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
11539,2022-12-25,CN,China,WPRO,40475477.0,62445650,5849.0,38995


In [8]:
covid_data[covid_data["New_cases"] == covid_data["New_cases"].min()]

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
50543,2023-08-20,PH,Philippines,WPRO,-65079.0,4108552,0.0,66646


Negative values in `New_cases` and `New_deaths`. On further investigation it appears that this might reflect corrections in historical data which can happen when a country has overestimated the number of cases/deaths.

## Nulls & duplicates

### - Nulls

In [9]:
covid_data.isna().sum()/len(covid_data)*100

Date_reported         0.000000
Country_code          0.416667
Country               0.000000
WHO_region            7.500000
New_cases            34.184727
Cumulative_cases      0.000000
New_deaths           52.776058
Cumulative_deaths     0.000000
dtype: float64

In [10]:
covid_data[covid_data['Country_code'].isna()]['Country'].unique()

array(['Namibia'], dtype=object)

In [11]:
covid_data[covid_data['Country'] == 'Namibia']['Country_code'].unique()

array([nan], dtype=object)

In [12]:
covid_data['Country_code'] = covid_data['Country_code'].fillna('NA')

The only country missing a `Country_code` was **Namibia**, whose ISO 3166-1 alpha-2 
code is ironically `NA` — the same string pandas interprets as a null value by default. 
Null replaced manually with the string `'NA'` using `.fillna('NA')`.

In [13]:
covid_data.isna().sum()

Date_reported            0
Country_code             0
Country                  0
WHO_region            5814
New_cases            26500
Cumulative_cases         0
New_deaths           40912
Cumulative_deaths        0
dtype: int64

In [14]:
covid_data[covid_data['WHO_region'].isna()].groupby('Country').count()

,Date_reported,Country_code,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths
Country,,,,,,,
Falkland Islands (Malvinas),323,323,0,57,323,0,323
Faroe Islands,323,323,0,87,323,14,323
French Guiana,323,323,0,151,323,92,323
Gibraltar,323,323,0,140,323,31,323
Guadeloupe,323,323,0,158,323,119,323
Guernsey,323,323,0,128,323,35,323
Holy See,323,323,0,8,323,0,323
Isle of Man,323,323,0,103,323,48,323
Jersey,323,323,0,151,323,68,323


In [15]:
covid_data['Country'] = covid_data['Country'].replace(to_replace='R�union', value='Réunion')
covid_data['Country'] = covid_data['Country'].replace(to_replace='Saint Barth�lemy', value='Saint Barthélemy')

In [16]:
region_mapping = {
    'Falkland Islands (Malvinas)': 'AMRO',
    'Faroe Islands': 'EURO',
    'French Guiana': 'AMRO',
    'Gibraltar': 'EURO',
    'Guadeloupe': 'AMRO',
    'Guernsey': 'EURO',
    'Holy See': 'EURO',
    'Isle of Man': 'EURO',
    'Jersey': 'EURO',
    'Liechtenstein': 'EURO',
    'Martinique': 'AMRO',
    'Mayotte': 'AFRO',
    'Pitcairn': 'WPRO',
    'Réunion': 'AFRO',
    'Saint Barthélemy': 'AMRO',
    'Saint Helena': 'AFRO',
    'Saint Martin (French part)': 'AMRO',
    'Saint Pierre and Miquelon': 'AMRO'
}

In [17]:
covid_data['WHO_region'] = covid_data['WHO_region'].fillna(covid_data['Country'].map(region_mapping))

Fill missing values in `WHO_region` using ***map.()*** built-in function

In [18]:
covid_data.isna().sum()/len(covid_data) * 100

Date_reported         0.000000
Country_code          0.000000
Country               0.000000
WHO_region            0.000000
New_cases            34.184727
Cumulative_cases      0.000000
New_deaths           52.776058
Cumulative_deaths     0.000000
dtype: float64

We continue to have Nulls in `New_cases` and `New_deaths`. 
This could be ***Structurally Missing Data (MNAR)*** if nulls represent days with zero 
cases/deaths, or ***Missing at Random (MAR)*** if they reflect reporting gaps. Since no clear assumption can be made without it affecting the data quality I've decided to create two new columns `New_cases_clean` and `New_deaths_clean` which will forward fill with previuos data.

In [19]:
covid_data[['New_cases_clean', 'New_deaths_clean']] = (covid_data.groupby('Country')[['New_cases', 'New_deaths']].ffill())

In [20]:
covid_data.isna().sum()/len(covid_data) *100

Date_reported         0.000000
Country_code          0.000000
Country               0.000000
WHO_region            0.000000
New_cases            34.184727
Cumulative_cases      0.000000
New_deaths           52.776058
Cumulative_deaths     0.000000
New_cases_clean       3.219814
New_deaths_clean      7.936017
dtype: float64

In [21]:
covid_data[covid_data['New_cases_clean'].isna()].groupby('Country').tail()

,Date_reported,Country_code,Country,WHO_region,New_cases,Cumulative_cases,New_deaths,Cumulative_deaths,New_cases_clean,New_deaths_clean
7,2020-02-23,AI,Anguilla,AMRO,NaN,0,NaN,0,NaN,NaN
8,2020-03-01,AI,Anguilla,AMRO,NaN,0,NaN,0,NaN,NaN
9,2020-03-08,AI,Anguilla,AMRO,NaN,0,NaN,0,NaN,NaN
10,2020-03-15,AI,Anguilla,AMRO,NaN,0,NaN,0,NaN,NaN
11,2020-03-22,AI,Anguilla,AMRO,NaN,0,NaN,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
77212,2020-04-19,XXL,International commercial vessel,OTHER,NaN,0,NaN,0,NaN,NaN
77213,2020-04-26,XXL,International commercial vessel,OTHER,NaN,0,NaN,0,NaN,NaN
77214,2020-05-03,XXL,International commercial vessel,OTHER,NaN,0,NaN,0,NaN,NaN
77215,2020-05-10,XXL,International commercial vessel,OTHER,NaN,0,NaN,0,NaN,NaN


Forward filling was applied per country to handle missing values in `New_cases` and `New_deaths` due to reporting delays. After cleaning, only a small percentage of NaNs remain (~3 for cases, ~8% for deaths). These remaining missing values correspond to the earliest dates in each country during the early pandemic period. Since the cumulative cases and deaths on those dates are 0, it is reasonable to assume that these NaNs represent true zero values rather than missing data.

### - Duplicates

In [22]:
covid_data.duplicated().sum()

np.int64(0)

No duplicated rows have been found

### Column dtypes

In [23]:
covid_data.dtypes

Date_reported         object
Country_code          object
Country               object
WHO_region            object
New_cases            float64
Cumulative_cases       int64
New_deaths           float64
Cumulative_deaths      int64
New_cases_clean      float64
New_deaths_clean     float64
dtype: object

New_cases and New_deaths are discrete by nature but appear as float, this might be due to missing values (NaN)

In [24]:
covid_data["Date_reported"] = pd.to_datetime(covid_data["Date_reported"])

In [25]:
covid_data.dtypes

Date_reported        datetime64[ns]
Country_code                 object
Country                      object
WHO_region                   object
New_cases                   float64
Cumulative_cases              int64
New_deaths                  float64
Cumulative_deaths             int64
New_cases_clean             float64
New_deaths_clean            float64
dtype: object

## Dataset adjustment for merge
- Date
- Column names
- Country

### - Date

In [26]:
covid_adjusted = covid_data[(covid_data["Date_reported"] >= '2021-01-31') & (covid_data["Date_reported"] <= '2023-12-31')]
print(covid_adjusted["Date_reported"].min())
print(covid_adjusted["Date_reported"].max())

2021-01-31 00:00:00
2023-12-31 00:00:00


With this adjustment we now have the same date range to perform our analysis correctly since the sarting and endind dates match.

### Column names

In [27]:
covid_adjusted.columns

Index(['Date_reported', 'Country_code', 'Country', 'WHO_region', 'New_cases',
       'Cumulative_cases', 'New_deaths', 'Cumulative_deaths',
       'New_cases_clean', 'New_deaths_clean'],
      dtype='object')

In [28]:
covid_adjusted = covid_adjusted.rename(columns={'Date_reported': 'Date'})
covid_adjusted.dtypes

Date                 datetime64[ns]
Country_code                 object
Country                      object
WHO_region                   object
New_cases                   float64
Cumulative_cases              int64
New_deaths                  float64
Cumulative_deaths             int64
New_cases_clean             float64
New_deaths_clean            float64
dtype: object

In [29]:
covid_adjusted.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36720 entries, 162 to 77405
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               36720 non-null  datetime64[ns]
 1   Country_code       36720 non-null  object        
 2   Country            36720 non-null  object        
 3   WHO_region         36720 non-null  object        
 4   New_cases          29543 non-null  float64       
 5   Cumulative_cases   36720 non-null  int64         
 6   New_deaths         22587 non-null  float64       
 7   Cumulative_deaths  36720 non-null  int64         
 8   New_cases_clean    36425 non-null  float64       
 9   New_deaths_clean   35125 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(2), object(3)
memory usage: 3.1+ MB


In [32]:
covid_adjusted.to_csv('../data/processed/covid_cleaned.csv', index=False)